<a href="https://colab.research.google.com/github/AhsanullahCS/FlyRank_Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AhsanullahCS/FlyRank_Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [4]:
%pip -q install duckdb

In [12]:
import duckdb

con = duckdb.connect()

con.execute("""
CREATE OR REPLACE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN ?
)
""", [HF_TOKEN])

print("DuckDB Hugging Face secret created.")

DuckDB Hugging Face secret created.


In [7]:
from google.colab import userdata
from huggingface_hub import whoami

HF_TOKEN = userdata.get("HF_TOKEN")

info = whoami(token=HF_TOKEN)

print("Logged in as:", info["name"])

Logged in as: Ahs1238


In [8]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

files = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

print("Access successful")
print("Number of files:", len(files))

Access successful
Number of files: 24


In [15]:
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print("File downloaded successfully")
print(file_path)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

File downloaded successfully
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [16]:
import duckdb

march_df = duckdb.query(f"""
    SELECT *
    FROM read_parquet('{file_path}')
    LIMIT 5
""").df()

march_df

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [17]:
print(march_df.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [18]:
result = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT
        CAST(client_hash_id AS VARCHAR) || '|' ||
        CAST(content_hash_id AS VARCHAR) || '|' ||
        CAST(report_date AS VARCHAR)
    ) AS distinct_grains
FROM read_parquet('{file_path}')
""").df()

result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grains
0,9841378,9841378


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [20]:
result = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet('{file_path}')
WHERE month = '2026-03'
""").df()

print(result)

   row_count first_date  last_date
0    9841378 2026-03-01 2026-03-31


In [21]:
result = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows
FROM read_parquet('{file_path}')
WHERE month = '2026-03'
""").df()

result

,total_rows,gsc_available_rows
0,9841378,3611061


In [23]:
import pandas as pd
df = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_users
FROM read_parquet('{file_path}')
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
""").df()

df["report_date"] = pd.to_datetime(df["report_date"])

df = df.sort_values(
    ["client_hash_id", "content_hash_id", "report_date"]
)

group_cols = ["client_hash_id", "content_hash_id"]

df["prev_gsc_impressions"] = (
    df.groupby(group_cols)["gsc_impressions"].shift(1)
)

df["prev_gsc_clicks"] = (
    df.groupby(group_cols)["gsc_clicks"].shift(1)
)

df["prev_gsc_avg_position"] = (
    df.groupby(group_cols)["gsc_avg_position"].shift(1)
)

df["prev_ga4_sessions"] = (
    df.groupby(group_cols)["ga4_sessions"].shift(1)
)

df["prev_ga4_users"] = (
    df.groupby(group_cols)["ga4_users"].shift(1)
)

df["next_day_gsc_clicks"] = (
    df.groupby(group_cols)["gsc_clicks"].shift(-1)
)

df["label_next_day_click"] = (
    df["next_day_gsc_clicks"] > 0
).astype(int)

feature_frame = df[
    [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "prev_gsc_impressions",
        "prev_gsc_clicks",
        "prev_gsc_avg_position",
        "prev_ga4_sessions",
        "prev_ga4_users",
        "label_next_day_click"
    ]
].dropna()

feature_frame.head(10)

,client_hash_id,content_hash_id,report_date,prev_gsc_impressions,prev_gsc_clicks,prev_gsc_avg_position,prev_ga4_sessions,prev_ga4_users,label_next_day_click
572513,client_08d2847f24cf89c1,content_0071174ee1f89d8a,2026-03-02,3.0,0.0,9.000000,0,0,0
131567,client_08d2847f24cf89c1,content_0071174ee1f89d8a,2026-03-03,1.0,0.0,11.000000,0,0,0
602188,client_08d2847f24cf89c1,content_0071174ee1f89d8a,2026-03-04,9.0,0.0,6.666667,0,0,0
503069,client_08d2847f24cf89c1,content_0071174ee1f89d8a,2026-03-05,4.0,0.0,7.250000,0,0,0
831724,client_08d2847f24cf89c1,content_0071174ee1f89d8a,2026-03-07,1.0,0.0,8.000000,0,0,0
776505,client_08d2847f24cf89c1,content_0071174ee1f89d8a,2026-03-08,2.0,0.0,14.500000,0,0,0
969985,client_08d2847f24cf89c1,content_0071174ee1f89d8a,2026-03-09,9.0,0.0,8.111111,0,0,0
1032711,client_08d2847f24cf89c1,content_0071174ee1f89d8a,2026-03-10,9.0,0.0,7.111111,0,0,0
1346426,client_08d2847f24cf89c1,content_0071174ee1f89d8a,2026-03-11,6.0,0.0,5.833333,0,0,0
1275235,client_08d2847f24cf89c1,content_0071174ee1f89d8a,2026-03-12,1.0,0.0,7.000000,0,0,0


In [24]:
leaky_df = feature_frame.copy()

# Deliberately create a leaked feature from the label
leaky_df["LEAKED_LABEL_FEATURE"] = leaky_df["label_next_day_click"]

leaky_df[
    ["LEAKED_LABEL_FEATURE", "label_next_day_click"]
].head(10)

,LEAKED_LABEL_FEATURE,label_next_day_click
572513,0,0
131567,0,0
602188,0,0
503069,0,0
831724,0,0
776505,0,0
969985,0,0
1032711,0,0
1346426,0,0
1275235,0,0


In [25]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X = leaky_df[["LEAKED_LABEL_FEATURE"]]
y = leaky_df["label_next_day_click"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = DecisionTreeClassifier(max_depth=2, random_state=42)

model.fit(X_train, y_train)

pred = model.predict(X_test)

leaky_accuracy = accuracy_score(y_test, pred)

print("Accuracy with leakage:", leaky_accuracy)

Accuracy with leakage: 1.0


In [26]:
honest_df = leaky_df.drop(
    columns=["LEAKED_LABEL_FEATURE"]
)

print("Leaked feature removed.")
print(honest_df.columns.tolist())

Leaked feature removed.
['client_hash_id', 'content_hash_id', 'report_date', 'prev_gsc_impressions', 'prev_gsc_clicks', 'prev_gsc_avg_position', 'prev_ga4_sessions', 'prev_ga4_users', 'label_next_day_click']


# W03 — Data Contract

## 1. Contract

### What one row means
One row represents one content item for one client on one report date.

### Which table I use
I use the `fact_content_daily_performance` table from the FlyRank internship warehouse.

### Time window
I use March 2026 (`month = '2026-03'`) as the development month.

### What I predict
I use whether the content item receives GSC clicks on the following day (`next_day_gsc_clicks > 0`) as a binary proxy for next-day search performance.

### What I deliberately exclude
I deliberately exclude future and same-day outcome information from the feature set because it would not be available at the decision moment and could cause data leakage.


## 2. Fields

### Features
I use five historical features:

1. Previous-day GSC impressions
2. Previous-day GSC clicks
3. Previous-day GSC average position
4. Previous-day GA4 sessions
5. Previous-day GA4 users

### Label
The label is `label_next_day_click`, which is 1 when the content item receives at least one GSC click on the following day, and 0 otherwise.

### Context
The context fields include:

- `client_hash_id`
- `content_hash_id`
- `report_date`
- `client_has_gsc`
- `client_has_ga4`
- `gsc_data_available`
- `ga4_data_available`

### Excluded
I exclude future and same-day outcome information from the features because it would not be known at the decision moment.


## 3. Verification and Features

### Verification results

**Grain:**  
The March dataset contains 9,841,378 rows and 9,841,378 distinct `(client_hash_id, content_hash_id, report_date)` combinations. This supports the claim that one row represents one content item for one client on one report date.

**Row count and date span:**  
The March 2026 slice contains 9,841,378 rows, covering 2026-03-01 through 2026-03-31.

**Availability:**  
There are 3,611,061 rows where `gsc_data_available IS TRUE`.


### Five features — available when?

1. **Previous-day GSC impressions** — knowable at the decision moment because the previous day's GSC data has already been collected.

2. **Previous-day GSC clicks** — knowable at the decision moment because they come from completed historical GSC data.

3. **Previous-day GSC average position** — knowable at the decision moment because it is calculated from the previous day's completed GSC observations.

4. **Previous-day GA4 sessions** — knowable at the decision moment because the previous day's GA4 session data is already available.

5. **Previous-day GA4 users** — knowable at the decision moment because the previous day's GA4 user data is already available.


### Leakage experiment

I deliberately created a `LEAKED_LABEL_FEATURE` by copying the next-day label directly into a feature. This caused the quick model's score to jump toward perfect accuracy because the model was given information that directly reveals the outcome.

This is data leakage because the leaked information would not be available at the decision moment.

I then removed the `LEAKED_LABEL_FEATURE` and kept only the five historical features. The honest feature set is the one that should be used for modeling.


## 4. Limitation

This analysis uses only the March 2026 slice, so one month may not capture longer-term trends, seasonality, or changes in content performance across different periods.


## 5. Self-check

- [x] Contract written in plain words
- [x] Unit of analysis defined
- [x] Warehouse table identified
- [x] Time window defined
- [x] Label/proxy defined
- [x] Deliberate exclusion defined
- [x] Grain verified
- [x] Row count and date span verified
- [x] Availability checked using `IS TRUE`
- [x] Five features created
- [x] Availability timing documented for each feature
- [x] Leakage experiment performed
- [x] Leaked feature removed
- [x] Limitation documented """

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.